# E08_SoccerArena — MA-POCA с самоигрой

**Урок:** 3.2 «MA-POCA и Self-Play» курса [cyber-unity-learn](https://rl-cuber-unity-code.com/courses/3-2)
и проект [/unity-projects/soccer](https://rl-cuber-unity-code.com/unity-projects/soccer)
**Что реализуется:** мультиагентный actor-critic с централизованным критиком и
контрфактическим базлайном; соперник — замороженный снимок политики из пула.
**Ожидаемый результат:** **≥ 60 %** побед над последним снимком на 20 матчах оценки.
**Время обучения:** около 60 минут на сид (`QUICK_RUN = True` — около пяти минут).

## Что здесь принципиально нового

Первая среда лаборатории, в которой **нарушено допущение стационарности**.
Во всех предыдущих семи средах мир был неизменен: одна и та же политика
в одном и том же состоянии получала один и тот же распределённый отклик.
Здесь мир для обучаемой команды включает трёх других агентов, и пока они
меняются, меняется и мир.

Отсюда три новых механизма, и каждый решает свою половину задачи.

**1. Общая награда и распределение заслуги.** Гол засчитан команде.
Обычный policy gradient усилит все действия обоих игроков одинаково —
включая бесполезное стояние в углу. MA-POCA вычитает **контрфактический
базлайн** $Q_i$: ожидаемый возврат команды при известных действиях всех,
кроме $i$-го. Что осталось — вклад именно $i$-го игрока.

**2. CTDE.** Критик и базлайн во время обучения видят команду целиком,
актор — только своё наблюдение. В ONNX уезжает **только актор**: у игрока
в сцене Unity нет наблюдений партнёра, и централизованных входов там взять
неоткуда.

**3. Self-Play.** Соперник берётся из пула замороженных снимков прошлых
версий политики. На протяжении блока шагов он не меняется — то есть среда
для обучаемой команды снова стационарна.

## Отсюда — главное следствие для чтения графиков

**Награда за матч перестаёт быть мерой прогресса.** Она измеряет силу
относительно текущего соперника, а тот усиливается вместе с политикой.
Идеально идущее обучение выглядит как награда, колеблющаяся около нуля.

Мерами прогресса служат **доля побед над последним снимком** и **ELO**.
Растущий ELO при стоящей на месте награде — нормальная картина self-play.

## Требования

```powershell
python scripts\build_env.py E08_SoccerArena
```

## 1. Окружение

In [ ]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import torch.nn as nn
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
DEVICE = resolve_device("auto")

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {DEVICE}")
print(f"сид           {SEED}")

## 2. Конфигурация

Значения совпадают с `configs/E08_SoccerArena__mapoca.yaml`: ноутбук и скрипт
обязаны обучать одинаково, иначе учебный и рабочий пути разойдутся.

Отличия от `E06_Hunter3D__ppo.ipynb`, и каждое имеет причину:

| Параметр | E06 (PPO) | E08 (MA-POCA) | Почему |
|---|---|---|---|
| `entropy_coef` | 0.01 | 0.005 | три ветки по три действия дают стартовую энтропию $3\ln 3 \approx 3.3$ нат — втрое больше, чем у одноветочной среды |
| `minibatch_size` | 256 переходов | 128 **шагов команды** | единица батча здесь — команда, а не отдельный агент |
| `baseline_coef` | — | 0.5 | у PPO контрфактического базлайна нет |
| блок `selfplay` | — | есть | пул снимков, ELO, темп смены соперника |

**Про `swap_every_steps` и `save_every_steps`.** Это два разных ритма.
Первый — как часто менять соперника: слишком часто и среда снова
нестационарна, слишком редко и политика переобучается под одного противника.
Второй — как часто пополнять пул: он задаёт, насколько «свежими» будут
соперники в среднем.

In [ ]:
QUICK_RUN = True   # True — быстрая проверка (~5 мин); False — полное обучение (~60 мин)

from labrl.algos.mapoca import MAPOCAConfig
from labrl.train.selfplay import SelfPlayConfig
from labrl.utils.schedules import LinearSchedule

TOTAL_STEPS = 40_000 if QUICK_RUN else 400_000

ALGO_CFG = MAPOCAConfig(
    gamma=0.99,
    gae_lambda=0.95,
    learning_rate=3e-4,
    clip_range=0.2,
    epochs=3,
    minibatch_size=128,      # шагов КОМАНДЫ, не отдельных переходов
    value_coef=0.5,
    baseline_coef=0.5,       # вес ошибки контрфактического базлайна
    entropy_coef=0.005,      # ниже, чем в E06: три ветки дают втрое больше энтропии
    max_grad_norm=0.5,
    normalize_advantage=True,
    target_kl=0.0,
)

TRAIN_CFG = SelfPlayConfig(
    total_steps=TOTAL_STEPS,
    rollout_steps=256,               # 256 шагов × 4 команды = 1024 шага команды
    swap_every_steps=5_000,          # как часто менять соперника
    save_every_steps=10_000,         # как часто пополнять пул снимков
    window=10,                       # размер пула
    play_against_latest_ratio=0.5,   # доля матчей против текущей политики
    elo_k=16.0,
    eval_every_steps=max(1, TOTAL_STEPS // 8),
    eval_matches=20,
)

# Расписание НЕ сжимается вместе с сокращённым прогоном (T-10).
LEARNING_RATE = LinearSchedule(start=3e-4, end=5e-5, decay_steps=400_000)

print(f"бюджет: {TOTAL_STEPS} шагов сбора, обновлений: {TOTAL_STEPS // TRAIN_CFG.rollout_steps}")
print(f"смен соперника: {TOTAL_STEPS // TRAIN_CFG.swap_every_steps}, "
      f"снимков в пуле к концу: {min(TRAIN_CFG.window, 1 + TOTAL_STEPS // TRAIN_CFG.save_every_steps)}")

## 3. Подключение среды

Здесь первое отличие от всех предыдущих ноутбуков. Unity добавляет к имени
поведения идентификатор команды (`BehaviorParameters.FullyQualifiedBehaviorName`),
поэтому одна сцена отдаёт **два** поведения:

```
E08_SoccerArena?team=0    команда West — обучается
E08_SoccerArena?team=1    команда East — соперник
```

`open_unity_env` сознательно отказывается выбирать «первое попавшееся»
из нескольких — self-play обязан обрабатываться явно, поэтому среда
открывается по полному имени команды.

Обёртка `TeamUnityEnv` трактует единицей шага **команду**, а не агента:
четверо игроков арены видят один мяч, и разложить их в четыре независимых
слота значит выбросить ровно ту связь, ради которой среда заведена.

In [ ]:
from labrl.envs.registry import get as get_env
from labrl.envs.team_env import TeamUnityEnv, team_behavior_name
from labrl.envs.unity_env import open_unity_env

ENV_ID = "E08_SoccerArena"
LEARNER_TEAM, OPPONENT_TEAM = 0, 1

BUILD = get_env(ENV_ID).build_path
if not BUILD.is_file():
    raise FileNotFoundError(f"нет билда {BUILD}\nСоберите: python scripts/build_env.py {ENV_ID}")

handle = open_unity_env(
    team_behavior_name(ENV_ID, LEARNER_TEAM),   # полное имя: '...?team=0'
    build_path=BUILD, seed=SEED, time_scale=20.0, no_graphics=True,
)
env = TeamUnityEnv(handle, ENV_ID, team_size=2, team_ids=(LEARNER_TEAM, OPPONENT_TEAM))
steps = env.reset()

print(handle.describe())
print(f"\nповедения: {sorted(env.behavior_of.values())}")
print(f"групп (арен): {env.num_groups}, игроков в команде: {env.team_size}")
print(f"агентов всего: {env.num_groups * env.team_size * 2}")

## 4. Инспекция пространств

Проверка размерностей против строки-контракта `ENV_SPEC.md`. Для этой среды
к обычным проверкам добавляется одна специфическая: **порядок тегов лучевого
сенсора у двух команд обязан быть зеркальным**. Проверить его из Python
напрямую нельзя (теги живут в сцене), но можно проверить следствие: обе
команды видят наблюдение одной и той же формы, и значит одна сеть управляет
обеими.

In [ ]:
OBS_DIM = env.obs_dim                       # склейка сенсоров: 56 + 8
BRANCHES = env.discrete_branches

print(f"obs_shapes:        {[tuple(o.shape) for o in env.spec.observation_specs]}")
print(f"имена сенсоров:    {[o.name for o in env.spec.observation_specs]}")
print(f"obs_dim (склейка): {OBS_DIM}")
print(f"дискретные ветки:  {BRANCHES}")
print(f"непрерывных:       {env.action_spec.continuous_size}")

# Значения из строки-контракта ENV_SPEC.md:
# <!-- validator: obs_size=8; discrete_branches=3,3,3; continuous_size=0; max_step=3000 -->
assert [tuple(o.shape) for o in env.spec.observation_specs] == [(56,), (8,)], (
    "порядок или размер сенсоров не тот; ML-Agents сортирует сенсоры по имени — "
    "RayPerceptionSensor перед VectorSensor_size8")
assert OBS_DIM == 64, f"obs_dim {OBS_DIM} != 64"
assert BRANCHES == (3, 3, 3), f"ветки {BRANCHES} != (3, 3, 3) (ENV_SPEC.md §5)"
assert env.action_spec.continuous_size == 0

# Обе команды обязаны иметь одинаковую спецификацию: иначе одна сеть
# не сможет управлять обеими, и self-play разваливается.
for team in (LEARNER_TEAM, OPPONENT_TEAM):
    spec = handle.env.behavior_specs[env.behavior_of[team]]
    assert [tuple(o.shape) for o in spec.observation_specs] == [(56,), (8,)]
    assert tuple(spec.action_spec.discrete_branches) == BRANCHES

print(f"\nobs_0 (лучи, 56):  {np.array2string(steps[0].obs[0, 0, :8], precision=2)} …")
print(f"obs_1 (вектор, 8): {np.array2string(steps[0].obs[0, 0, 56:], precision=2)}")
print("  obs_1: [0:2] своя позиция, [2:4] мяч относительно себя,")
print("         [4:6] своя скорость, [6] косинус курса, [7] доля времени")
print("\nразмерности и порядок сенсоров совпадают с ENV_SPEC.md")

## 5. Нейросети

Три сети, потому что у них три разные роли:

| Сеть | Что видит | Уходит в ONNX |
|---|---|---|
| актор $\pi(a_i \mid o_i)$ | своё наблюдение (64) | **да** |
| критик $V(o_1 \dots o_n)$ | наблюдения всей команды | нет |
| базлайн $Q_i(o_1 \dots o_n, a_{-i})$ | наблюдения всех + действия всех, кроме $i$-го | нет |

Архитектуру задаёте вы: алгоритм принимает любой `nn.Module`, соблюдающий
формы (требование 8.4). Ниже — рабочий вариант, который можно править
прямо здесь.

**Почему критик устроен на внимании, а не на склейке наблюдений.**
Склейка фиксирует число агентов навсегда и заставляет учить каждую
перестановку игроков отдельно. Внимание принимает множество любого размера
и по построению не зависит от порядка — оба свойства покрыты тестами
`python/tests/test_mapoca.py`.

In [ ]:
from labrl.algos.mapoca import CentralizedCritic, CounterfactualBaseline
from labrl.nets.categorical_policy import MultiBranchCategoricalPolicy

EMBED_DIM, NUM_HEADS, ATT_HIDDEN = 64, 4, 128


class MyActor(MultiBranchCategoricalPolicy):
    """Децентрализованный актор: obs (B, 64) -> логиты (B, 9)."""

    def __init__(self, obs_dim: int, branches: tuple[int, ...], hidden: int = 256):
        super().__init__(obs_dim, branches, (hidden, hidden), "relu")


policy_net = MyActor(OBS_DIM, BRANCHES)
value_net = CentralizedCritic(OBS_DIM, EMBED_DIM, NUM_HEADS, ATT_HIDDEN)
baseline_net = CounterfactualBaseline(OBS_DIM, BRANCHES, EMBED_DIM, NUM_HEADS, ATT_HIDDEN)

with torch.no_grad():
    probe_obs = torch.zeros(3, env.team_size, OBS_DIM)
    probe_act = torch.zeros(3, env.team_size, len(BRANCHES), dtype=torch.long)
    probe_active = torch.ones(3, env.team_size)

    assert policy_net(torch.zeros(3, OBS_DIM)).shape == (3, sum(BRANCHES))
    assert value_net(probe_obs, probe_active).shape == (3,)
    assert baseline_net(probe_obs, probe_act, probe_active).shape == (3, env.team_size)

print(f"параметров актора:   {sum(p.numel() for p in policy_net.parameters()):,}")
print(f"параметров критика:  {sum(p.numel() for p in value_net.parameters()):,}")
print(f"параметров базлайна: {sum(p.numel() for p in baseline_net.parameters()):,}")
print("\nв ONNX уедет только актор — у игрока в Unity нет наблюдений партнёра")

## 6. Теория: контрфактический базлайн

Обычное преимущество отвечает на вопрос «насколько ход событий оказался
лучше ожидаемого»:

$$A = G^{\lambda} - V(s)$$

Для команды этого мало: вопрос стоит иначе — «насколько лучше стало
**из-за игрока $i$**». MA-POCA отвечает контрфактическим базлайном:

$$Q_i = Q\big(o_1 \dots o_n,\; a_1 \dots a_n \setminus a_i\big),
\qquad A_i = G^{\lambda} - Q_i$$

Всё, что объясняется действиями остальных, уже сидит в $Q_i$ и вычитается.
Игрок, чьё действие ничего не изменило, получает $A_i \approx 0$
и не усиливается.

**Два условия, без которых это не работает.**

1. $Q_i$ **не должен** зависеть от $a_i$ — иначе это обычный $Q$, и разность
   перестаёт измерять вклад. В коде это строка с `eye`: для сущности $j = i$
   берётся кодировщик **без** действия.
2. Через $A_i$ **не должен** идти градиент к базлайну (`detach`) — иначе метод
   начнёт занижать $Q_i$, чтобы «увеличить» преимущество.

Оба условия проверены тестом `test_baseline_ignores_own_action_but_uses_others`.

Цель обучения — $\lambda$-возврат **команды**, где награда складывается как

$$r_{\text{команды}} = r_{\text{группы}} + \textstyle\sum_i r_i$$

причём `group_reward` берётся **один раз**: Unity сообщает его одинаковым
у всех членов группы.

Ниже — исходный код обновления, тот самый, что выполняется при обучении.

In [ ]:
import inspect

from labrl.algos.mapoca import MAPOCA

print(inspect.getsource(MAPOCA.update))

## 7. Цикл обучения

Соперником управляет отдельный модуль той же архитектуры — **копия**,
а не ссылка: «замороженный» соперник, меняющийся вместе с обучаемой
политикой, не заморожен ни в каком смысле, и весь пул снимков теряет смысл.

In [ ]:
from labrl.logging.run_dir import create_run_dir
from labrl.logging.tb_logger import TBLogger
from labrl.train.selfplay import train_selfplay

algo = MAPOCA(policy_net, value_net, baseline_net, ALGO_CFG, device=DEVICE, seed=SEED)
run = create_run_dir(ENV_ID, "mapoca", SEED)
run.write_env_info({**handle.env_info(), "num_groups": env.num_groups,
                    "team_size": env.team_size, "seed": SEED,
                    "total_steps": TOTAL_STEPS, "quick": QUICK_RUN})
logger = TBLogger(run.tb)

print(f"каталог прогона: {run.root}\n")

result = train_selfplay(
    env=env, algo=algo, logger=logger, cfg=TRAIN_CFG, seed=SEED,
    learner_team=LEARNER_TEAM, opponent_team=OPPONENT_TEAM,
    lr_schedule=LEARNING_RATE,
    on_eval=lambda step, r, s: print(f"  шаг {step:>7}: награда {r:>7.3f}, побед {s:.0%}"),
)

missing = logger.missing_required_tags()
logger.dump_metrics(run.metrics_json, extra={
    "seed": SEED, "elo_history": result.elo_history,
    "missing_required_tags": list(missing)})
logger.close()

print(f"\nматчей: {len(result.match_returns)}, обновлений: {result.updates}")
print(f"ELO к концу: {result.final_elo:.0f}")
print(f"время: {result.wall_time:.1f} с")
print(f"обязательные теги схемы 11.2 без данных: {missing or 'нет'}")

## 8. Оценка и кривые обучения

Три графика, и **средний из них — не мера прогресса**. Награда за матч
измеряет силу относительно текущего соперника, а тот усиливается вместе
с политикой: идеально идущее обучение выглядит как награда около нуля.

Смотреть надо на долю побед над последним снимком и на ELO.

In [ ]:
import matplotlib
matplotlib.use("Agg")   # ноутбук должен исполняться и без графического бэкенда
import matplotlib.pyplot as plt

returns = np.array(result.match_returns)
window = max(1, len(returns) // 50)
smoothed = np.convolve(returns, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

steps = [s for s, _, _ in result.eval_history]
wins = [w for _, _, w in result.eval_history]

axes[0].plot(steps, wins, marker="o", color="tab:green", label="доля побед")
axes[0].axhline(0.6, color="k", ls="--", lw=1, label="критерий приёмки")
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title("Eval/Success Rate — мера качества")
axes[0].set_xlabel("шаг сбора"); axes[0].grid(alpha=0.3); axes[0].legend()

elo_steps = [s for s, _ in result.elo_history]
elo = [e for _, e in result.elo_history]
axes[1].plot(elo_steps, elo, marker="o", color="tab:blue")
axes[1].set_title("Custom/ELO — сила относительно истории")
axes[1].set_xlabel("шаг сбора"); axes[1].grid(alpha=0.3)

axes[2].plot(smoothed, color="tab:orange")
axes[2].set_title(f"Награда за матч (среднее по {window}) — НЕ мера прогресса")
axes[2].set_xlabel("матч"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"итоговая оценка: побед {result.last_eval_success:.0%}, "
      f"награда {result.last_eval_reward:+.3f}, ELO {result.final_elo:.0f}")

# Критерий приёмки проверяется только на ПОЛНОМ бюджете: сокращённый прогон —
# смоук-тест конвейера, на нём метод не обязан обучиться.
THRESHOLD = 0.60   # блок success_criteria конфига, метрика Eval/Success Rate

if QUICK_RUN:
    print(f"быстрый прогон: критерий приёмки ({THRESHOLD:.0%}) не проверяется — "
          "проверялась работоспособность конвейера")
else:
    assert result.last_eval_success >= THRESHOLD, (
        f"не достигнут критерий приёмки: {result.last_eval_success:.0%} < {THRESHOLD:.0%}")
    print(f"критерий приёмки достигнут: {result.last_eval_success:.0%} >= {THRESHOLD:.0%}")

## 9. Экспорт в ONNX и верификация

Экспортируется **только актор**, стратегия `categorical`: `discrete_actions`
получается сэмплированием из softmax, `deterministic_discrete_actions` —
argmax по каждой ветке. Контракт отдаёт `(batch, 3)` **целых** индексов —
Unity читает этот тензор как `Tensor<int>`, и приведение к float ломает
инференс, оставляя ONNX формально валидным
(`docs/04_ONNX_CONTRACT.md`, §2).

In [ ]:
import onnxruntime as ort

from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

# Наблюдения из реального распределения среды (требование 10.5), а не шум.
obs_shapes = [tuple(o.shape) for o in env.spec.observation_specs]
flat = []
steps_now = env.reset()
while sum(len(f) for f in flat) < 64:
    for team_step in steps_now.values():
        present = team_step.active.reshape(-1)
        flat.append(team_step.obs.reshape(env.num_groups * env.team_size, -1)[present])
    steps_now = env.step({
        team: algo.deterministic_action(
            steps_now[team].obs.reshape(env.num_groups * env.team_size, -1)
        ).reshape(env.num_groups, env.team_size, -1)
        for team in env.team_ids
    })

stacked = np.concatenate(flat, axis=0)[:64]
sample, offset = [], 0
for shape in obs_shapes:
    sample.append(stacked[:, offset:offset + shape[0]])
    offset += shape[0]

spec = ActionSpecLite(discrete_branches=BRANCHES, continuous_size=0)
policy = algo.policy_module()
onnx_path = export_policy_to_onnx(policy, spec, obs_shapes, run.onnx / "policy.onnx",
                                  strategy="categorical")

verification = verify_onnx_model(onnx_path, policy, spec, obs_shapes, sample_obs=sample)
print(verification.report())
verification.raise_if_failed()

# Python и граф обязаны давать одно и то же детерминированное действие.
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
graph_action = session.run(
    ["deterministic_discrete_actions"],
    {"obs_0": sample[0], "obs_1": sample[1],
     "action_masks": np.ones((64, sum(BRANCHES)), dtype=np.float32),
     "recurrent_in": np.zeros((64, 1, 0), dtype=np.float32)},
)[0]
python_action = algo.deterministic_action(np.concatenate(sample, axis=1))
mismatch = int((graph_action != python_action).sum())
print(f"\nрасхождений действия ONNX и Python: {mismatch} из {python_action.size}")
assert mismatch == 0, "оценка в Python и граф расходятся — инференс в Unity будет другим"

if not QUICK_RUN:
    # Модель проекта обновляется только полным прогоном: быстрый затёр бы её
    # смоук-версией (T-14 в docs/07_TROUBLESHOOTING.md).
    model_dest = (REPO_ROOT / "unity" / "MLAgentsLab" / "Assets" / "Envs" / ENV_ID /
                  "Models" / f"{ENV_ID}_mapoca.onnx")
    model_dest.parent.mkdir(parents=True, exist_ok=True)
    model_dest.write_bytes(onnx_path.read_bytes())
    print(f"модель скопирована в проект Unity: {model_dest.relative_to(REPO_ROOT)}")
else:
    print("быстрый прогон: модель в проект Unity НЕ копируется (T-14)")

env.close()

## 10. Возврат в Unity

### Автоматически (то, чем проверяется приёмка по 10.6)

```powershell
python scripts\check_inference.py --config configs\E08_SoccerArena__mapoca.yaml --python-reward <оценка из раздела 8>
```

### Вручную (чтобы увидеть глазами)

1. Открыть `unity/MLAgentsLab` в Unity 6000.5.4f1.
2. Открыть `Assets/Envs/E08_SoccerArena/Scenes/E08_SoccerArena.unity`.
3. Выделить модель в Project → **Tools → RL → Check ONNX Contract**.
4. Назначить модель **всем четырём** игрокам арены: `Model` = импортированная
   модель, `Behavior Type` = `Inference Only`, галочка `Deterministic Inference`.
5. Play: обе команды играют одной и той же политикой — так и задумано.
   Перспектива задаётся не весами, а зеркальным порядком `DetectableTags`
   и командной системой координат в наблюдении (`ENV_SPEC.md`, §4).

### Если что-то идёт не так

| Симптом | Где искать |
|---|---|
| одна команда явно сильнее другой при одинаковой модели | порядок `DetectableTags` не зеркальный либо поле несимметрично |
| игроки едут в свои ворота | знак зеркалирования в `CollectObservations` |
| `Custom/Baseline Spread` был нулём всё обучение | базлайн не различает вклад игроков — MA-POCA выродился в PPO |

Общий список причин расхождения Python и Unity — в
`docs/07_TROUBLESHOOTING.md`.

### Ручная проверка среды с клавиатуры

`Behavior Type` = `Heuristic Only` у игрока `TeamWest/Player_0`
первой арены. Клавиши: **W/S** — вперёд/назад, **A/D** — боком,
**Q/E** — поворот.